# Long-Horizon Task Execution: Decomposition and Replanning

Long-horizon tasks — those requiring many steps over extended time — expose failure modes that short-horizon agents never encounter: accumulated error, goal drift, environmental change, and the need to replan when execution deviates from the original plan.

## Task Decomposition Strategies

Breaking a complex goal into manageable subtasks is the first challenge in long-horizon execution. Two strategies:

**Top-down decomposition**: start with the high-level goal, recursively decompose into subtasks until each is small enough to execute directly. Clean and predictable but requires knowing the full decomposition upfront, which is often impossible for complex tasks.

**Online decomposition**: generate the next subtask based on current state. More adaptive but harder to track progress toward the overall goal and more prone to goal drift.

Hybrid: decompose into large phases upfront, then decompose each phase online. This gives predictable structure while allowing adaptation within phases.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional
import json

@dataclass
class Subtask:
    id: str
    description: str
    dependencies: list
    status: str = 'pending'  # pending, running, done, failed
    result: str = ''
    attempts: int = 0

@dataclass
class TaskPlan:
    goal: str
    subtasks: list
    checkpoints: list = field(default_factory=list)

    def next_executable(self) -> Optional[Subtask]:
        done_ids = {s.id for s in self.subtasks if s.status == 'done'}
        for s in self.subtasks:
            if s.status == 'pending' and all(d in done_ids for d in s.dependencies):
                return s
        return None

    def is_complete(self) -> bool:
        return all(s.status == 'done' for s in self.subtasks)

    def has_failures(self) -> bool:
        return any(s.status == 'failed' for s in self.subtasks)

class GoalConditionedAgent:
    def __init__(self, executor_fn: Callable, replan_fn: Callable, max_retries: int = 2):
        self.executor = executor_fn
        self.replan = replan_fn
        self.max_retries = max_retries
        self.execution_log = []

    def execute_plan(self, plan: TaskPlan) -> dict:
        while not plan.is_complete():
            next_task = plan.next_executable()
            if not next_task:
                if plan.has_failures():
                    # Attempt replanning
                    new_tasks = self.replan(plan)
                    if new_tasks:
                        plan.subtasks.extend(new_tasks)
                        # Reset failed tasks that were replanned
                        for s in plan.subtasks:
                            if s.status == 'failed':
                                s.status = 'skipped'
                        continue
                break
            next_task.status = 'running'
            next_task.attempts += 1
            result = self.executor(next_task)
            if result['success']:
                next_task.status = 'done'
                next_task.result = result['output']
            elif next_task.attempts >= self.max_retries:
                next_task.status = 'failed'
            else:
                next_task.status = 'pending'  # retry
            self.execution_log.append({'task': next_task.id, 'status': next_task.status,
                                        'attempt': next_task.attempts})
        done = sum(1 for s in plan.subtasks if s.status == 'done')
        return {'complete': plan.is_complete(), 'done': done,
                'total': len(plan.subtasks), 'log': self.execution_log}

rng = random.Random(42)
def mock_executor(task):
    # 80% success rate
    success = rng.random() < 0.8
    return {'success': success, 'output': f'Result of {task.id}' if success else ''}

def mock_replanner(plan):
    failed = [s for s in plan.subtasks if s.status == 'failed']
    if failed:
        return [Subtask(f'{s.id}_alt', f'Alternative approach for {s.description}', s.dependencies)
                for s in failed[:1]]
    return []

plan = TaskPlan(
    goal='Write and publish a technical blog post',
    subtasks=[
        Subtask('outline', 'Create post outline', []),
        Subtask('draft', 'Write first draft', ['outline']),
        Subtask('review', 'Review and edit draft', ['draft']),
        Subtask('format', 'Format for publication', ['draft']),
        Subtask('publish', 'Publish the post', ['review', 'format']),
    ]
)
agent = GoalConditionedAgent(mock_executor, mock_replanner)
result = agent.execute_plan(plan)
print(f'Plan execution: {result["done"]}/{result["total"]} tasks done')
print(f'Complete: {result["complete"]}')
for log in result['log']:
    print(f'  {log["task"]}: {log["status"]} (attempt {log["attempt"]})')

## Checkpointing for Resilience

Long-running agent tasks need checkpointing — saving progress so execution can resume after failure:

1. After each completed subtask, serialize the plan state to durable storage
2. On restart, load the last checkpoint and skip already-completed tasks
3. For tasks with side effects, include idempotency checks before re-executing

Checkpointing turns a catastrophic failure (agent crashes after 45 minutes) into a recoverable one (resume from the last checkpoint). This is standard practice for ML training jobs — it should be equally standard for long-running agents.